# Working with Datasets — User Guide

`StarLayerDataset` extends rdflib's `Dataset`: a single store holding several independently addressable named graphs, each an RDF-1.2-aware `StarLayerGraph` under the hood. This is a general graph-storage feature, not RDF-1.2-specific — see the [Graphs guide](02-graphs.ipynb) for the triple-term/reification/literal semantics each individual graph carries, and the [serialization formats guide](02b-graphs-serialization-formats.ipynb) for the dataset-capable formats (`trig12`, `trix12`, `nq12`) used here.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells from top to bottom — later cells reuse variables from earlier ones.

In [1]:
from starlayergraph import StarLayerDataset, Namespace

EX = Namespace("http://example.org/")

## Creating and populating named graphs

`dataset.graph(<name>)` returns (creating if needed) the graph for that name. `dataset.query()` runs SPARQL across the whole dataset, with `GRAPH ?g { ... }` patterns scoping to individual graphs the way plain SPARQL expects.

In [2]:
ds = StarLayerDataset()
ds.bind("ex", EX)

g1 = ds.graph(EX.graph1)
g2 = ds.graph(EX.graph2)

g1.add((EX.bob, EX.knows, EX.carol))
g1.add((EX.claim, EX.source, EX.wikipedia))
g2.add((EX.bob, EX.likes, EX.dana))

print("g1 triple count:", len(g1))
print("g2 triple count:", len(g2))
print("graph names:", sorted(str(c.identifier) for c in ds.graphs()))

g1 triple count: 2
g2 triple count: 1
graph names: ['http://example.org/graph1', 'http://example.org/graph2', 'urn:x-rdflib:default']


In [3]:
# GRAPH ?g scopes the pattern to each named graph in turn, same as plain SPARQL
rows = ds.query('''
PREFIX ex: <http://example.org/>
SELECT ?g ?s ?p ?o WHERE {
  GRAPH ?g { ?s ?p ?o }
}
ORDER BY ?g ?s
''')
for row in rows:
    print(row)

(rdflib.term.URIRef('http://example.org/graph1'), rdflib.term.URIRef('http://example.org/bob'), rdflib.term.URIRef('http://example.org/knows'), rdflib.term.URIRef('http://example.org/carol'))
(rdflib.term.URIRef('http://example.org/graph1'), rdflib.term.URIRef('http://example.org/claim'), rdflib.term.URIRef('http://example.org/source'), rdflib.term.URIRef('http://example.org/wikipedia'))
(rdflib.term.URIRef('http://example.org/graph2'), rdflib.term.URIRef('http://example.org/bob'), rdflib.term.URIRef('http://example.org/likes'), rdflib.term.URIRef('http://example.org/dana'))


## The dataset's own default graph, and dataset-wide serialization

Adding directly to the dataset (`ds.add(...)`, no graph specified) writes to the dataset's own default graph, separate from any named graph. Serializing the whole dataset with a dataset format (`trig12`) shows every named graph, each in its own `GRAPH` block — see the [serialization formats guide](02b-graphs-serialization-formats.ipynb) for `trig12`'s single-graph counterparts and the rest of the format set.

In [4]:
print(ds.serialize(format="trig12"))

@prefix ex: <http://example.org/> .

GRAPH <http://example.org/graph1> {
    ex:bob ex:knows ex:carol .

    ex:claim ex:source ex:wikipedia .
}

GRAPH <http://example.org/graph2> {
    ex:bob ex:likes ex:dana .
}



## Further work

- **Blank nodes on a remote backend.** The examples above use the default in-memory store. Writing blank nodes to a dataset backed by a remote SPARQL store (Oxigraph, Fuseki) goes through the same skolemization handling documented in [`starlayergraph.md`](../../packages/graph/docs/starlayergraph.md)'s "Blank nodes against a remote store" section — see [the backend guide](06-backend-graph-databases.ipynb) for the full picture.